In [63]:
# Импорт необходимых библиотек
import pymorphy3  # для морфологического анализа русского языка
import csv  # для работы с CSV файлами
from collections import Counter  # для подсчета частотности
import re  # для работы с регулярными выражениями

In [64]:
# Инициализация морфологического анализатора
morph = pymorphy3.MorphAnalyzer()

In [65]:
def load_text(file_path):
    """Загружает текст из файла"""
    with open(file_path, 'r', encoding='ANSI') as file:
        return file.read()

In [66]:
def load_roles(file_path):
    """Загрузка ролей из файла, пропуская комментарии и пустые строки"""
    with open(file_path, 'r', encoding='utf-8') as file:
        return set(line.strip() for line in file if line.strip() and not line.startswith('#'))

In [67]:
def load_framebank_data(file_path):
    """Загрузка аннотированных примеров из FrameBank в формате CSV"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as file:
        reader = csv.DictReader(file, delimiter='\t')
        for row in reader:
            data.append(row)
    return data

In [68]:
def extract_sentences_with_ids(text):
    """Извлекает предложения из текста с сохранением их нумерации"""
    sentences = {}
    current_sentence = []
    current_id = None

In [69]:
sentences = {}

In [70]:
current_sentence = []

In [71]:
current_id = None

In [74]:
def extract_sentences_with_ids(text):
    """Извлекает предложения из текста с сохранением их нумерации"""
    sentences = {}
    current_sentence = []
    current_id = None

    for line in text.splitlines():
        # Поиск номера предложения в начале строки
        match = re.match(r'^\s*(\d+)\s', line)
        if match:
            if current_sentence and current_id is not None:
                sentences[current_id] = ' '.join(current_sentence)
            current_id = int(match.group(1))
            current_sentence = []

        # Извлечение токенов из XML-разметки
        tokens = re.findall(r'<w>.*?<ana.*?/>(.*?)</w>', line)
        if tokens:
            current_sentence.extend(tokens)

    # Сохранение последнего предложения
    if current_sentence and current_id is not None:
        sentences[current_id] = ' '.join(current_sentence)

    return sentences


In [75]:
def search_by_semantic_role(framebank_data, roles):
    """Поиск контекстов по заданному набору семантических ролей"""
    result = {}
    for row in framebank_data:
        role = row['Role']
        if role in roles:
            key = f"{row['ExIndex']} ({row['Phrase']})"
            result[key] = row
    return result, len(result)

In [76]:
def search_specific_role(framebank_data, target_role):
    """Поиск контекстов по конкретной семантической роли"""
    result = {}
    for row in framebank_data:
        if row['Role'] == target_role:
            key = f"{row['ExIndex']} ({row['Phrase']})"
            result[key] = row
    return result, len(result)

In [77]:
def search_by_token(sentences, token):
    """Ищет контексты по заданному токену (точное совпадение)"""
    result = {sid: sentence for sid, sentence in sentences.items() if token in sentence}
    return result, len(result)

In [78]:
def search_by_lemma(sentences, lemma):
    """Ищет контексты по заданной лемме (нормальной форме слова)"""
    result = {}
    for sid, sentence in sentences.items():
        words = sentence.split()
        for word in words:
            parsed = morph.parse(word)[0]
            if parsed.normal_form == lemma:
                result[sid] = sentence
                break
    return result, len(result)

In [79]:
def search_by_semantic_tag(text, tag):
    """Ищет контексты по заданному семантическому тегу в XML-разметке"""
    sentences = {}
    current_sentence = []
    current_id = None
    tag_found = False
    
    for line in text.splitlines():
        # Обработка начала нового предложения
        match = re.match(r'^\s*(\d+)\s', line)
        if match:
            if current_sentence and current_id is not None and tag_found:
                sentences[current_id] = ' '.join(current_sentence)
            current_id = int(match.group(1))
            current_sentence = []
            tag_found = False

        # Поиск тега в разметке
        if re.search(rf'<ana [^>]*{re.escape(tag)}', line):
            tag_found = True

        # Извлечение токенов
        tokens = re.findall(r'<w>.*?<ana.*?/>(.*?)</w>', line)
        if tokens:
            current_sentence.extend(tokens)

    # Сохранение последнего предложения
    if current_sentence and current_id is not None and tag_found:
        sentences[current_id] = ' '.join(current_sentence)

    return sentences, len(sentences)

In [80]:
def search_by_tag_combination(text, tags):
    """Ищет контексты по заданной комбинации тегов"""
    sentences = {}
    current_sentence = []
    current_id = None
    tags_found = set()

    for line in text.splitlines():
        # Обработка начала нового предложения
        match = re.match(r'^\s*(\d+)\s', line)
        if match:
            if current_sentence and current_id is not None and tags_found == set(tags):
                sentences[current_id] = ' '.join(current_sentence)
            current_id = int(match.group(1))
            current_sentence = []
            tags_found = set()

        # Поиск всех тегов в разметке
        for tag in tags:
            if re.search(rf'<ana [^>]*{re.escape(tag)}', line):
                tags_found.add(tag)

        # Извлечение токенов
        tokens = re.findall(r'<w>.*?<ana.*?/>(.*?)</w>', line)
        if tokens:
            current_sentence.extend(tokens)

    # Сохранение последнего предложения
    if current_sentence and current_id is not None and tags_found == set(tags):
        sentences[current_id] = ' '.join(current_sentence)

    return sentences, len(sentences)


In [81]:
def count_frequencies(sentences, text):
    """Подсчитывает частотность различных элементов в тексте"""
    tokens = []
    lemmas = []
    tags = []
    combinations = []

    # Подсчет токенов и лемм
    for sentence in sentences.values():
        words = re.findall(r'\b\w+\b', sentence)
        tokens.extend(words)
        lemmas.extend([morph.parse(word)[0].normal_form for word in words])

    # Подсчет тегов и их комбинаций
    for line in text.splitlines():
        tags.extend(re.findall(r'gr=\'(.*?)\'|sem=\'(.*?)\'', line))
        combinations.extend(re.findall(r'gr=\'(.*?)\' sem=\'(.*?)\'', line))

    return len(tokens), len(lemmas), len(tags), len(combinations)

In [82]:
def save_summary(token_count, lemma_count, tag_count, combination_count, file_name):
    """Сохраняет статистику в файл"""
    with open(file_name, 'w', encoding='ANSI') as file:
        file.write(f"Токенов: {token_count}\n")
        file.write(f"Лемм: {lemma_count}\n")
        file.write(f"Тегов: {tag_count}\n")
        file.write(f"Комбинаций тегов: {combination_count}\n")

In [89]:

def save_contexts(contexts, count, file_name):
    """Сохраняет найденные контексты в файл"""
    with open(file_name, 'w', encoding='ANSI') as file:
        file.write(f"Количество: {count}\n\n")
        for sid, context in contexts.items():
            file.write(f"{sid}: {context}\n")

In [90]:
def save_results(results, count, file_name):
    """Сохраняет результаты поиска по семантическим ролям в файл"""
    with open(file_name, 'w', encoding='utf-8') as file:
        file.write(f"Количество найденных примеров: {count}\n\n")
        for key, row in results.items():
            file.write(f"{key} -> {row['Role']} ({row['Phrase']})\n")

In [ ]:
# Основной код
# Загрузка необходимых файлов
file_path =  "C:\\Users\\79040\\OneDrive\\Рабочий стол\\MSP\\2 семестр HW\\HW_search_context\\RNC_Subcorpus\\RNC_Subcorpus\\RNC_Subcorpus\\instrumenty\\metla.txt"
framebank_file_path = "C:\\Users\\79040\\OneDrive\\Рабочий стол\\MSP\\2 семестр HW\\HW_search_context\\framebank_anno_ex_circ.txt"
role_file_path = "C:\\Users\\79040\\OneDrive\\Рабочий стол\\MSP\\2 семестр HW\\HW_search_context\\framebank_roles.txt"
text = load_text(file_path)
sentences = extract_sentences_with_ids(text)
roles = load_roles(role_file_path)
framebank_data = load_framebank_data(framebank_file_path)

In [91]:

# Задание с framework
specific_role = 'степень'
specific_role_contexts, specific_role_count = search_specific_role(framebank_data, specific_role)
save_results(specific_role_contexts, specific_role_count, 'specific_role_results.txt')
role_contexts, role_count = search_by_semantic_role(framebank_data, roles)
save_results(role_contexts, role_count, 'all_roles_results.txt')

# Задание а: Поиск по токену
token = 'метлой'
token_contexts, token_count = search_by_token(sentences, token)
save_contexts(token_contexts, token_count, 'token_results.txt')

# Задание б: Поиск по лемме
lemma = 'метла'
lemma_contexts, lemma_count = search_by_lemma(sentences, lemma)
save_contexts(lemma_contexts, lemma_count, 'lemma_results.txt')

# Задание в: Поиск по семантическому тегу
tag = 'r:spec'
tag_contexts, tag_count = search_by_semantic_tag(text, tag)
save_contexts(tag_contexts, tag_count, 'tag_results.txt')

# Задание г: Поиск по комбинации тегов
tags = ['r:concr', 't:tool:instr']
combination_contexts, combination_count = search_by_tag_combination(text, tags)
save_contexts(combination_contexts, combination_count, 'combination_results.txt')

# Задание д: Подсчёт частотности токенов, лемм, тегов и комбинаций тегов
token_count, lemma_count, tag_count, combination_count = count_frequencies(sentences, text)
save_summary(token_count, lemma_count, tag_count, combination_count, 'summary.txt')

print("Операции завершены. Результаты сохранены в файлы.")

Операции завершены. Результаты сохранены в файлы.
